# AudioTools Function Testing Notebook

This notebook demonstrates and tests all major audiotools functions using real audio files from the tests directory.

## Setup and Imports

In [ ]:
import audiotools
from audiotools import AudioSignal
from audiotools.data import transforms as tfm
from audiotools.core import util
from audiotools import metrics
import torch
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
util.seed(42)

print(f"AudioTools version: {audiotools.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 1. Loading Audio Files

Test different ways to load audio from the test directory.

In [ ]:
# Load speech audio
speech = AudioSignal("tests/audio/spk/f10_script4_produced.wav")
print(f"Speech - Sample rate: {speech.sample_rate} Hz, Duration: {speech.duration:.2f}s, Shape: {speech.audio_data.shape}")

# Load noise audio
noise = AudioSignal("tests/audio/nz/f5_script2_ipad_balcony1_room_tone.wav")
print(f"Noise - Sample rate: {noise.sample_rate} Hz, Duration: {noise.duration:.2f}s, Shape: {noise.audio_data.shape}")

# Load impulse response
ir = AudioSignal("tests/audio/ir/h179_Bar_1txts.wav")
print(f"IR - Sample rate: {ir.sample_rate} Hz, Duration: {ir.duration:.2f}s, Shape: {ir.audio_data.shape}")

In [ ]:
# Load with offset and duration
speech_excerpt = AudioSignal("tests/audio/spk/f10_script4_produced.wav", offset=2, duration=3)
print(f"Speech excerpt - Duration: {speech_excerpt.duration:.2f}s")

In [ ]:
# Load from CSV sources
speech_sources = util.read_sources(["tests/audio/spk.csv"])
noise_sources = util.read_sources(["tests/audio/noises.csv"])
ir_sources = util.read_sources(["tests/audio/irs.csv"])

print(f"Found {len(speech_sources)} speech files")
print(f"Found {len(noise_sources)} noise files")
print(f"Found {len(ir_sources)} IR files")

## 2. Visualization

Test waveform and spectrogram plotting functions.

In [ ]:
# Waveform plot
fig, ax = plt.subplots(figsize=(12, 3))
speech_excerpt.waveplot(ax=ax)
plt.title("Speech Waveform")
plt.tight_layout()
plt.show()

In [ ]:
# Spectrogram plot
fig, ax = plt.subplots(figsize=(12, 4))
speech_excerpt.specshow(ax=ax)
plt.title("Speech Spectrogram")
plt.tight_layout()
plt.show()

In [ ]:
# Combined waveform and spectrogram
fig, axes = speech_excerpt.wavespec()
plt.suptitle("Speech Waveform and Spectrogram")
plt.tight_layout()
plt.show()

## 3. Basic Signal Operations

Test cloning, copying, resampling, and mono conversion.

In [ ]:
# Clone signal
speech_clone = speech_excerpt.clone()
print(f"Cloned signal shape: {speech_clone.audio_data.shape}")

# Resample to different sample rate
speech_resampled = speech_excerpt.clone().resample(16000)
print(f"Resampled to 16kHz: {speech_resampled.sample_rate} Hz, {speech_resampled.audio_data.shape}")

# Convert to mono
speech_mono = speech_excerpt.clone().to_mono()
print(f"Mono signal shape: {speech_mono.audio_data.shape}")

In [ ]:
# Extract excerpt
random_excerpt = speech.excerpt(duration=2.0)
print(f"Random excerpt duration: {random_excerpt.duration:.2f}s")

# Extract salient (loudest) excerpt
salient_excerpt = speech.salient_excerpt(duration=2.0)
print(f"Salient excerpt duration: {salient_excerpt.duration:.2f}s")

In [ ]:
# Padding and trimming
padded = speech_excerpt.clone().zero_pad(100, 100)
print(f"Padded signal length: {padded.signal_length} samples")

trimmed = padded.trim(100, 100)
print(f"Trimmed signal length: {trimmed.signal_length} samples")

## 4. DSP Operations

Test filtering, windowing, and other DSP functions.

In [ ]:
# Low-pass filter
speech_lowpass = speech_excerpt.clone().low_pass(4000)
print(f"Low-pass filtered at 4kHz")

# High-pass filter
speech_highpass = speech_excerpt.clone().high_pass(200)
print(f"High-pass filtered at 200Hz")

# Visualize filtered signals
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")
speech_lowpass.specshow(ax=axes[1])
axes[1].set_title("Low-pass (4kHz)")
speech_highpass.specshow(ax=axes[2])
axes[2].set_title("High-pass (200Hz)")
plt.tight_layout()
plt.show()

In [ ]:
# Windowing
windows = speech_excerpt.clone().windows(window_duration=0.1, hop_duration=0.05)
print(f"Windows shape: {windows.shape}")

# Collect windows
window_list = speech_excerpt.clone().collect_windows(window_duration=0.1, hop_duration=0.05)
print(f"Number of windows: {len(window_list)}")
print(f"First window shape: {window_list[0].audio_data.shape}")

## 5. Effects

Test volume, pitch, time stretch, and other effects.

In [ ]:
# Volume change
speech_louder = speech_excerpt.clone().volume_change(db=6)
speech_quieter = speech_excerpt.clone().volume_change(db=-6)
print(f"Volume changed by +6dB and -6dB")

# Normalize to target loudness
speech_normalized = speech_excerpt.clone().normalize(db=-20)
print(f"Normalized to -20 LUFS")

In [ ]:
# Pitch shift
speech_pitched_up = speech_excerpt.clone().pitch_shift(n_semitones=4)
speech_pitched_down = speech_excerpt.clone().pitch_shift(n_semitones=-4)
print(f"Pitch shifted by +4 and -4 semitones")

In [ ]:
# Time stretch
speech_faster = speech_excerpt.clone().time_stretch(factor=1.2)
speech_slower = speech_excerpt.clone().time_stretch(factor=0.8)
print(f"Original duration: {speech_excerpt.duration:.2f}s")
print(f"Faster (1.2x): {speech_faster.duration:.2f}s")
print(f"Slower (0.8x): {speech_slower.duration:.2f}s")

In [ ]:
# Mix signals with SNR control
noise_excerpt = noise.excerpt(duration=speech_excerpt.duration)
speech_with_noise = speech_excerpt.clone().mix(noise_excerpt, snr=10)
print(f"Mixed speech with noise at SNR=10dB")

# Visualize mixed signal
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Clean Speech")
noise_excerpt.specshow(ax=axes[1])
axes[1].set_title("Noise")
speech_with_noise.specshow(ax=axes[2])
axes[2].set_title("Speech + Noise (SNR=10dB)")
plt.tight_layout()
plt.show()

In [ ]:
# Apply impulse response (convolution)
speech_reverb = speech_excerpt.clone().apply_ir(ir, drr=0)
print(f"Applied impulse response (reverb)")

# Visualize reverb effect
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Dry Speech")
speech_reverb.specshow(ax=axes[1])
axes[1].set_title("Speech with Reverb")
plt.tight_layout()
plt.show()

In [ ]:
# Codec simulation
speech_codec = speech_excerpt.clone().apply_codec("mp3", bitrate="64k")
print(f"Applied MP3 codec at 64kbps")

In [ ]:
# Distortion effects
speech_clipped = speech_excerpt.clone().clip_distortion(percentile=0.2)
print(f"Applied clipping distortion")

speech_quantized = speech_excerpt.clone().quantization(channels=8)
print(f"Applied 8-bit quantization")

speech_mulaw = speech_excerpt.clone().mulaw_quantization(channels=8)
print(f"Applied mu-law quantization")

## 6. STFT and Spectral Operations

Test STFT, mel spectrograms, and spectral manipulations.

In [ ]:
# STFT
speech_stft = speech_excerpt.clone()
speech_stft.stft()
print(f"STFT shape: {speech_stft.stft_data.shape}")
print(f"Magnitude shape: {speech_stft.magnitude.shape}")
print(f"Phase shape: {speech_stft.phase.shape}")

# Inverse STFT
speech_stft.istft()
print(f"Reconstructed audio shape: {speech_stft.audio_data.shape}")

In [ ]:
# Mel spectrogram
mel_spec = speech_excerpt.mel_spectrogram(n_mels=80)
print(f"Mel spectrogram shape: {mel_spec.shape}")

# MFCC
mfcc = speech_excerpt.mfcc(n_mfcc=13)
print(f"MFCC shape: {mfcc.shape}")

# Visualize mel spectrogram
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(mel_spec[0].cpu().numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(im, ax=ax)
ax.set_title("Mel Spectrogram")
ax.set_xlabel("Time")
ax.set_ylabel("Mel Bin")
plt.tight_layout()
plt.show()

In [ ]:
# Phase manipulation
speech_phase_shifted = speech_excerpt.clone().shift_phase(shift=np.pi/4)
print(f"Phase shifted by π/4")

speech_phase_inverted = speech_excerpt.clone().invert_phase()
print(f"Phase inverted")

speech_phase_corrupt = speech_excerpt.clone().corrupt_phase(scale=0.5)
print(f"Phase corrupted")

In [ ]:
# Masking
speech_freq_masked = speech_excerpt.clone().mask_frequencies(fmin_hz=1000, fmax_hz=3000)
print(f"Masked frequencies 1-3kHz")

speech_time_masked = speech_excerpt.clone().mask_timesteps(tmin_s=0.5, tmax_s=1.5)
print(f"Masked time 0.5-1.5s")

speech_mag_masked = speech_excerpt.clone().mask_low_magnitudes(db_cutoff=-40)
print(f"Masked low magnitudes below -40dB")

# Visualize masking
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
speech_excerpt.specshow(ax=axes[0, 0])
axes[0, 0].set_title("Original")
speech_freq_masked.specshow(ax=axes[0, 1])
axes[0, 1].set_title("Frequency Masked (1-3kHz)")
speech_time_masked.specshow(ax=axes[1, 0])
axes[1, 0].set_title("Time Masked (0.5-1.5s)")
speech_mag_masked.specshow(ax=axes[1, 1])
axes[1, 1].set_title("Magnitude Masked (<-40dB)")
plt.tight_layout()
plt.show()

## 7. Loudness Operations

Test loudness measurement and normalization.

In [ ]:
# Measure loudness
loudness = speech_excerpt.loudness()
print(f"Loudness: {loudness:.2f} LUFS")

# Normalize to target loudness
speech_norm_20 = speech_excerpt.clone().normalize(db=-20)
loudness_norm = speech_norm_20.loudness()
print(f"Normalized loudness: {loudness_norm:.2f} LUFS")

# Ensure max of audio
speech_safe = speech_excerpt.clone().ensure_max_of_audio()
print(f"Max value after ensuring: {speech_safe.audio_data.abs().max():.4f}")

## 8. Transforms

Test data augmentation transforms.

In [ ]:
# Room impulse response transform
rir_transform = tfm.RoomImpulseResponse(sources=["tests/audio/irs.csv"])
kwargs = rir_transform.instantiate(seed=42, signal=speech_excerpt)
speech_rir = rir_transform(speech_excerpt.clone(), **kwargs)
print(f"Applied RIR transform")

# Background noise transform
bg_noise_transform = tfm.BackgroundNoise(sources=["tests/audio/noises.csv"], snr=(5, 20))
kwargs = bg_noise_transform.instantiate(seed=42, signal=speech_excerpt)
speech_bg_noise = bg_noise_transform(speech_excerpt.clone(), **kwargs)
print(f"Applied background noise transform")

In [ ]:
# Compose multiple transforms
composed_transform = tfm.Compose([
    tfm.VolumeChange(db=(-6, 6)),
    tfm.RoomImpulseResponse(sources=["tests/audio/irs.csv"]),
    tfm.BackgroundNoise(sources=["tests/audio/noises.csv"], snr=(5, 15)),
    tfm.LowPass(cutoff=(4000, 8000)),
])

kwargs = composed_transform.instantiate(seed=42, signal=speech_excerpt)
speech_augmented = composed_transform(speech_excerpt.clone(), **kwargs)
print(f"Applied composed transform")

# Visualize augmentation pipeline
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")
speech_augmented.specshow(ax=axes[1])
axes[1].set_title("Augmented (Volume + Reverb + Noise + LowPass)")
plt.tight_layout()
plt.show()

In [ ]:
# Test various single transforms
transforms_to_test = [
    ("Clipping Distortion", tfm.ClippingDistortion(perc=(0.1, 0.3))),
    ("Quantization", tfm.Quantization(channels=(4, 8))),
    ("Equalizer", tfm.Equalizer(eq_amount=(1.0, 10.0))),
    ("Low Pass", tfm.LowPass(cutoff=(2000, 8000))),
    ("High Pass", tfm.HighPass(cutoff=(50, 500))),
    ("Frequency Mask", tfm.FrequencyMask(f_width=(0.0, 0.3))),
    ("Time Mask", tfm.TimeMask(t_width=(0.0, 0.1))),
]

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.flatten()

speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")

for i, (name, transform) in enumerate(transforms_to_test, start=1):
    kwargs = transform.instantiate(seed=42, signal=speech_excerpt)
    transformed = transform(speech_excerpt.clone(), **kwargs)
    transformed.specshow(ax=axes[i])
    axes[i].set_title(name)
    
plt.tight_layout()
plt.show()

## 9. Metrics

Test audio quality metrics.

In [ ]:
# Create a degraded version
speech_degraded = speech_excerpt.clone()
speech_degraded = speech_degraded.low_pass(3000)
speech_degraded = speech_degraded.apply_codec("mp3", bitrate="64k")

# Resample both to 16kHz for metrics
speech_ref = speech_excerpt.clone().resample(16000)
speech_deg = speech_degraded.resample(16000)

print("Computing audio quality metrics...")
print("(This may take a moment)\n")

In [ ]:
# STOI (Short-Time Objective Intelligibility)
try:
    stoi_score = metrics.stoi(speech_ref, speech_deg)
    print(f"STOI: {stoi_score:.4f}")
except Exception as e:
    print(f"STOI error: {e}")

In [ ]:
# PESQ (Perceptual Evaluation of Speech Quality)
try:
    pesq_score = metrics.pesq(speech_ref, speech_deg)
    print(f"PESQ: {pesq_score:.4f}")
except Exception as e:
    print(f"PESQ error: {e}")

In [ ]:
# ViSQOL (Virtual Speech Quality Objective Listener)
try:
    visqol_score = metrics.visqol(speech_ref, speech_deg)
    print(f"ViSQOL: {visqol_score:.4f}")
except Exception as e:
    print(f"ViSQOL error: {e}")

In [ ]:
# Distance metrics
l1_loss = metrics.distance.L1Loss()(speech_ref, speech_deg)
print(f"L1 Loss: {l1_loss.item():.6f}")

sisdr_loss = metrics.distance.SISDRLoss()(speech_ref, speech_deg)
print(f"SI-SDR Loss: {sisdr_loss.item():.6f}")

## 10. Batch Processing

Test batching multiple signals together.

In [ ]:
# Create multiple excerpts
excerpts = [speech.excerpt(duration=2.0) for _ in range(5)]
print(f"Created {len(excerpts)} excerpts")

# Batch signals
batched = util.collate(excerpts)
print(f"Batched shape: {batched.audio_data.shape}")
print(f"Batch size: {batched.batch_size}")

In [ ]:
# Apply batch transform
batch_transform = tfm.VolumeChange(db=(-6, 6))
kwargs = batch_transform.instantiate(seed=42, signal=batched)
batched_transformed = batch_transform(batched.clone(), **kwargs)
print(f"Transformed batch shape: {batched_transformed.audio_data.shape}")

## 11. Save/Export

Test saving audio to different formats.

In [ ]:
# Save to WAV
speech_excerpt.write("/tmp/test_output.wav")
print("Saved to /tmp/test_output.wav")

# Save to MP3
speech_excerpt.write("/tmp/test_output.mp3")
print("Saved to /tmp/test_output.mp3")

# Verify files were created
import os
print(f"\nWAV exists: {os.path.exists('/tmp/test_output.wav')}")
print(f"MP3 exists: {os.path.exists('/tmp/test_output.mp3')}")

## 12. Advanced: Custom Processing Pipeline

Demonstrate a complete audio processing pipeline.

In [ ]:
def audio_augmentation_pipeline(signal, seed=None):
    """Complete audio augmentation pipeline."""
    if seed is not None:
        util.seed(seed)
    
    # Clone input
    output = signal.clone()
    
    # 1. Normalize loudness
    output = output.normalize(db=-20)
    
    # 2. Add room acoustics
    rir_transform = tfm.RoomImpulseResponse(sources=["tests/audio/irs.csv"])
    kwargs = rir_transform.instantiate(seed=seed, signal=output)
    output = rir_transform(output, **kwargs)
    
    # 3. Add background noise
    bg_transform = tfm.BackgroundNoise(sources=["tests/audio/noises.csv"], snr=(10, 20))
    kwargs = bg_transform.instantiate(seed=seed, signal=output)
    output = bg_transform(output, **kwargs)
    
    # 4. Apply codec
    output = output.apply_codec("mp3", bitrate="128k")
    
    # 5. Final normalize
    output = output.normalize(db=-20)
    
    return output

# Apply pipeline
speech_processed = audio_augmentation_pipeline(speech_excerpt, seed=42)

# Visualize results
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
speech_excerpt.specshow(ax=axes[0])
axes[0].set_title("Original")
speech_processed.specshow(ax=axes[1])
axes[1].set_title("Processed (Normalized + Reverb + Noise + Codec)")
plt.tight_layout()
plt.show()

print(f"Original loudness: {speech_excerpt.loudness():.2f} LUFS")
print(f"Processed loudness: {speech_processed.loudness():.2f} LUFS")

## Summary

This notebook demonstrated:

1. **Loading**: Multiple ways to load audio from files and CSV sources
2. **Visualization**: Waveform and spectrogram plotting
3. **Basic Operations**: Cloning, resampling, mono conversion, excerpting
4. **DSP**: Filtering, windowing, overlap-add
5. **Effects**: Volume, pitch shift, time stretch, mixing, reverb, codec simulation, distortion
6. **Spectral**: STFT, mel spectrograms, MFCC, phase manipulation, masking
7. **Loudness**: Measurement and normalization
8. **Transforms**: Data augmentation for training
9. **Metrics**: Quality assessment (STOI, PESQ, ViSQOL, SI-SDR)
10. **Batching**: Processing multiple signals together
11. **Export**: Saving to various formats
12. **Pipelines**: Building complete processing workflows

All functions were tested with real audio files from the tests directory!